# Customer Feature Engineering

## Notebook purpose

This notebook creates leakage-safe customer features for return-risk modelling.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
FEATURES_DIR = DATA_DIR / "features"

FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature output directory: {FEATURES_DIR}")

Project root: C:\Users\ankur\OneDrive\Desktop\E-Commerce Project
Feature output directory: C:\Users\ankur\OneDrive\Desktop\E-Commerce Project\data\features


# Load Customer And Order Data

In [2]:
customers_df = pd.read_csv(DATA_DIR / "customer_master.csv")
orders_df = pd.read_csv(
    DATA_DIR / "ecommerce_sales_customer_analytics_150k.csv"
)

print(f"Customer rows: {len(customers_df):,}")
print(f"Order rows:    {len(orders_df):,}")

Customer rows: 25,000
Order rows:    138,116


# Prepare Order Timestamps And Return Labels

In [3]:
orders_df["order_timestamp"] = pd.to_datetime(
    orders_df["order_date"].astype(str)
    + " "
    + orders_df["order_time"].astype(str),
    errors="coerce"
)

orders_df["return_label"] = (
    orders_df["return_status"] == "Returned"
).astype(int)

invalid_timestamps = orders_df["order_timestamp"].isna().sum()

print(f"Invalid order timestamps: {invalid_timestamps:,}")

if invalid_timestamps > 0:
    raise ValueError(
        "Order timestamps could not be created. Check order_date and order_time."
    )

Invalid order timestamps: 0


In [4]:
orders_df.head()

,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,customer_type,customer_city,customer_state,customer_country,region,customer_postal_code,payment_method,payment_status,currency,shipping_method,warehouse,delivery_days,estimated_delivery_days,delivery_status,return_status,return_reason,customer_rating,review_sentiment,customer_review,marketing_channel,campaign_name,coupon_code,loyalty_points_earned,loyalty_points_redeemed,quantity,gross_sales,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count,order_timestamp,return_label
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,Loyal,Lake Williamberg,Texas,USA,South,86040,Digital Wallet,Paid,USD,Standard,WH-003,3.00,3.00,On Time,NaN,NaN,3.50,Positive,Satisfied with the purchase.,Direct,Default_Campaign,NaN,94,44,5,"1,350.19",477.89,61.07,12.03,945.40,588.33,345.04,36.50,"12,459.68",True,11,2023-11-06 16:37:47,0
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,Loyal,Kristyport,Baden-Württemberg,Germany,South,62538,Debit Card,Pending,EUR,Economy,WH-005,10.00,10.00,On Time,NaN,NaN,3.60,Positive,Good value for money.,Direct,Default_Campaign,NaN,201,171,8,"3,144.64","1,456.06",320.83,9.00,"2,018.41","2,031.88",-22.47,-1.11,"13,032.48",True,11,2025-12-24 01:22:36,0
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,Loyal,Singletonhaven,New York,USA,East,19993,Cash on Delivery,Paid,USD,Express,WH-015,3.00,3.00,On Time,NaN,NaN,4.30,Positive,Good product. Works as expected.,YouTube,NaN,NaN,46,16,5,522.81,97.30,29.79,13.05,468.35,257.73,197.57,42.18,"6,159.44",True,6,2021-07-05 14:24:21,0
3,ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,Loyal,Wigginsstad,North Carolina,USA,South,66329,Debit Card,Paid,USD,Express,WH-010,2.00,2.00,On Time,NaN,NaN,3.50,Positive,Satisfied with the purchase.,Email Marketing,NaN,NaN,54,10,2,544.62,53.34,34.39,20.85,546.52,277.84,247.83,45.35,"5,638.30",True,7,2023-01-21 07:20:26,0
4,ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,Loyal,New Michaelton,Gujarat,India,West,77306,Digital Wallet,Paid,INR,Standard,WH-013,6.00,5.00,Delayed,NaN,NaN,3.00,Neutral,Mixed feelings about this purchase.,Direct,NaN,NaN,278,147,6,"2,474.52",133.63,421.35,23.03,"2,785.27","1,231.88","1,530.36",54.94,"13,240.10",True,8,2022-05-13 09:46:21,0


# Select Safe Static Customer Attributes

In [5]:
customer_static_columns = [
    "customer_id",
    "customer_age",
    "gender",
    "customer_segment",
    "customer_state",
    "customer_country",
    "region",
    "customer_acquisition_cost",
]

customer_static_df = customers_df[customer_static_columns].copy()

print("Static customer columns:")
print(customer_static_df.columns.tolist())

Static customer columns:
['customer_id', 'customer_age', 'gender', 'customer_segment', 'customer_state', 'customer_country', 'region', 'customer_acquisition_cost']


# Group Simultaneous Orders Safely


In [6]:
history_base_df = orders_df[
    [
        "order_id",
        "customer_id",
        "order_timestamp",
        "net_sales",
        "discount_amount",
        "return_label",
    ]
].copy()

timestamp_history_df = (
    history_base_df
    .groupby(
        ["customer_id", "order_timestamp"],
        as_index=False
    )
    .agg(
        orders_at_timestamp=("order_id", "count"),
        net_sales_at_timestamp=("net_sales", "sum"),
        discount_at_timestamp=("discount_amount", "sum"),
        returns_at_timestamp=("return_label", "sum"),
    )
    .sort_values(["customer_id", "order_timestamp"])
    .reset_index(drop=True)
)

display(timestamp_history_df.head())

,customer_id,order_timestamp,orders_at_timestamp,net_sales_at_timestamp,discount_at_timestamp,returns_at_timestamp
0,CUST-000001,2021-02-09 13:02:39,1,913.70,48.79,0
1,CUST-000001,2021-07-26 08:27:56,1,"3,422.82",426.57,0
2,CUST-000001,2021-08-29 12:21:15,1,54.78,5.62,0
3,CUST-000001,2021-09-30 12:21:16,1,"1,258.95",252.94,0
4,CUST-000001,2022-12-05 23:39:53,1,455.29,0.00,0


# Create Prior-Order Historical Features

In [7]:
customer_groups = timestamp_history_df.groupby(
    "customer_id",
    group_keys=False
)

timestamp_history_df["prior_order_count"] = customer_groups[
    "orders_at_timestamp"
].transform(
    lambda series: series.cumsum().shift(fill_value=0)
)

timestamp_history_df["prior_total_spend"] = customer_groups[
    "net_sales_at_timestamp"
].transform(
    lambda series: series.cumsum().shift(fill_value=0)
)

timestamp_history_df["prior_total_discount"] = customer_groups[
    "discount_at_timestamp"
].transform(
    lambda series: series.cumsum().shift(fill_value=0)
)

timestamp_history_df["prior_return_count"] = customer_groups[
    "returns_at_timestamp"
].transform(
    lambda series: series.cumsum().shift(fill_value=0)
)

timestamp_history_df["previous_order_timestamp"] = customer_groups[
    "order_timestamp"
].shift()

timestamp_history_df["prior_average_order_value"] = np.where(
    timestamp_history_df["prior_order_count"] > 0,
    timestamp_history_df["prior_total_spend"]
    / timestamp_history_df["prior_order_count"],
    np.nan,
)

timestamp_history_df["prior_return_rate"] = np.where(
    timestamp_history_df["prior_order_count"] > 0,
    timestamp_history_df["prior_return_count"]
    / timestamp_history_df["prior_order_count"],
    np.nan,
)

timestamp_history_df["prior_average_discount"] = np.where(
    timestamp_history_df["prior_order_count"] > 0,
    timestamp_history_df["prior_total_discount"]
    / timestamp_history_df["prior_order_count"],
    np.nan,
)

timestamp_history_df["days_since_previous_order"] = (
    timestamp_history_df["order_timestamp"]
    - timestamp_history_df["previous_order_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

timestamp_history_df["is_first_order"] = (
    timestamp_history_df["prior_order_count"] == 0
).astype(int)

display(timestamp_history_df.head(10))

,customer_id,order_timestamp,orders_at_timestamp,net_sales_at_timestamp,discount_at_timestamp,returns_at_timestamp,prior_order_count,prior_total_spend,prior_total_discount,prior_return_count,previous_order_timestamp,prior_average_order_value,prior_return_rate,prior_average_discount,days_since_previous_order,is_first_order
0,CUST-000001,2021-02-09 13:02:39,1,913.70,48.79,0,0,0.00,0.00,0,NaT,NaN,NaN,NaN,NaN,1
1,CUST-000001,2021-07-26 08:27:56,1,"3,422.82",426.57,0,1,913.70,48.79,0,2021-02-09 13:02:39,913.70,0.00,48.79,166.81,0
2,CUST-000001,2021-08-29 12:21:15,1,54.78,5.62,0,2,"4,336.52",475.36,0,2021-07-26 08:27:56,"2,168.26",0.00,237.68,34.16,0
3,CUST-000001,2021-09-30 12:21:16,1,"1,258.95",252.94,0,3,"4,391.30",480.98,0,2021-08-29 12:21:15,"1,463.77",0.00,160.33,32.00,0
4,CUST-000001,2022-12-05 23:39:53,1,455.29,0.00,0,4,"5,650.25",733.92,0,2021-09-30 12:21:16,"1,412.56",0.00,183.48,431.47,0
5,CUST-000001,2024-11-16 02:45:06,1,99.01,0.00,1,5,"6,105.54",733.92,0,2022-12-05 23:39:53,"1,221.11",0.00,146.78,711.13,0
6,CUST-000001,2025-05-21 05:55:06,1,425.37,9.24,0,6,"6,204.55",733.92,1,2024-11-16 02:45:06,"1,034.09",0.17,122.32,186.13,0
7,CUST-000001,2025-08-06 05:09:21,1,"3,097.56",402.57,0,7,"6,629.92",743.16,1,2025-05-21 05:55:06,947.13,0.14,106.17,76.97,0
8,CUST-000001,2025-12-25 11:41:32,1,197.36,20.02,0,8,"9,727.48","1,145.73",1,2025-08-06 05:09:21,"1,215.93",0.12,143.22,141.27,0
9,CUST-000002,2021-04-27 00:56:30,1,"2,156.17",115.64,0,0,0.00,0.00,0,NaT,NaN,NaN,NaN,NaN,1


# Attach Historical Features To Individual Orders

In [8]:
customer_history_features = timestamp_history_df[
    [
        "customer_id",
        "order_timestamp",
        "prior_order_count",
        "prior_total_spend",
        "prior_total_discount",
        "prior_return_count",
        "prior_average_order_value",
        "prior_return_rate",
        "prior_average_discount",
        "days_since_previous_order",
        "is_first_order",
    ]
]

customer_features_df = history_base_df[
    ["order_id", "customer_id", "order_timestamp"]
].merge(
    customer_history_features,
    on=["customer_id", "order_timestamp"],
    how="left",
    validate="m:1",
)

print(f"Customer feature rows: {len(customer_features_df):,}")
print(f"Unique order IDs:      {customer_features_df['order_id'].nunique():,}")

Customer feature rows: 138,116
Unique order IDs:      138,116


# Add Static Customer Attributes

In [9]:
customer_features_df = customer_features_df.merge(
    customer_static_df,
    on="customer_id",
    how="left",
    validate="m:1",
)

missing_customer_attributes = (
    customer_features_df["customer_age"].isna().sum()
)

print(f"Orders without static customer attributes: {missing_customer_attributes:,}")

Orders without static customer attributes: 0


# Validate Feature Table

In [10]:
validation_summary = pd.Series({
    "total_rows": len(customer_features_df),
    "unique_order_ids": customer_features_df["order_id"].nunique(),
    "duplicate_order_ids": customer_features_df["order_id"].duplicated().sum(),
    "first_orders": customer_features_df["is_first_order"].sum(),
    "first_orders_with_nonzero_history": (
        customer_features_df.loc[
            customer_features_df["is_first_order"] == 1,
            "prior_order_count"
        ] > 0
    ).sum(),
    "contains_return_label": "return_label" in customer_features_df.columns,
})

display(validation_summary)

assert validation_summary["total_rows"] == validation_summary["unique_order_ids"]
assert validation_summary["duplicate_order_ids"] == 0
assert validation_summary["first_orders_with_nonzero_history"] == 0
assert validation_summary["contains_return_label"] is False

total_rows                           138116
unique_order_ids                     138116
duplicate_order_ids                       0
first_orders                          24911
first_orders_with_nonzero_history         0
contains_return_label                 False
dtype: object

# Review Example Customer Features

In [11]:
example_customer_id = customer_features_df[
    "customer_id"
].value_counts().index[0]

example_customer_history = (
    customer_features_df[
        customer_features_df["customer_id"] == example_customer_id
    ]
    .sort_values("order_timestamp")
    [
        [
            "order_id",
            "order_timestamp",
            "prior_order_count",
            "prior_total_spend",
            "prior_return_count",
            "prior_return_rate",
            "days_since_previous_order",
            "is_first_order",
        ]
    ]
)

display(example_customer_history)

,order_id,order_timestamp,prior_order_count,prior_total_spend,prior_return_count,prior_return_rate,days_since_previous_order,is_first_order
13758,ORD-142673,2021-04-11 23:46:06,0,0.00,0,NaN,NaN,1
8327,ORD-632172,2022-03-22 21:34:30,1,994.11,0,0.00,344.91,0
54355,ORD-798614,2022-06-01 17:23:41,2,"1,530.12",0,0.00,70.83,0
53937,ORD-417451,2023-02-12 00:10:55,3,"3,637.10",0,0.00,255.28,0
108049,ORD-779625,2023-03-19 17:17:39,4,"4,438.15",1,0.25,35.71,0
14891,ORD-857046,2023-04-04 04:30:31,5,"5,385.79",1,0.20,15.47,0
43882,ORD-164116,2023-06-10 23:10:38,6,"6,313.90",1,0.17,67.78,0
53053,ORD-515401,2023-08-30 00:43:05,7,"7,519.91",1,0.14,80.06,0
104358,ORD-894716,2024-03-13 20:38:56,8,"7,947.27",1,0.12,196.83,0
126168,ORD-728876,2024-04-18 00:34:24,9,"8,568.34",1,0.11,35.16,0


# Saving Customer Feature Table


In [12]:
output_path = FEATURES_DIR / "customer_features.csv"

customer_features_df.to_csv(output_path, index=False)

print(f"Saved customer features to: {output_path}")
print(f"Output shape: {customer_features_df.shape}")

Saved customer features to: C:\Users\ankur\OneDrive\Desktop\E-Commerce Project\data\features\customer_features.csv
Output shape: (138116, 19)
